<a href="https://colab.research.google.com/github/JSMultani/CS5841/blob/main/p4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P4 — Transfer Learning & Fine-Tuning (CPU-Friendly)
## Data Set: Oxford-IIIT Pet
- #### Feature Extraction vs Fine-Tuning | Performance Comparison
### Models: VGG16 • ResNet50 • DenseNet121
#### Solution Notebook

**Hardware assumption:** CPU-only laptops/PC are acceptable (training time may vary).  
**Dataset:** Oxford-IIIT Pet (37 classes)  
**Recommended settings:** `IMG_SIZE=(128,128)`, `BATCH_SIZE=32`, `EPOCHS=5-6` (CPU-friendly)

---

This solution notebook provides a clean reference implementation for:
- Building a reproducible `tf.data` pipeline
- Training **frozen-backbone** models (feature extraction)
- Running **one controlled fine-tuning** experiment
- Comparing results (accuracy, training time, parameter counts)


## Q0 — Setup (Ungraded)
#### Import libraries, set seeds, and verify TensorFlow / TFDS.

In [1]:
import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
from tensorflow.keras.applications import resnet, vgg16, densenet

print("TensorFlow:", tf.__version__)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

TensorFlow: 2.20.0


## ✅ Student Instructions (Start Here)

Your work begins in the **next code cells (Q1–Q9)** and continues by answering questions in the **Markdown cells (Q10–Q13)**. These correspond to the questions listed in the assignment description on Canvas. Complete each cell by following the instructions provided in the **preceding Markdown cells**.

Tasks:
- This assignment focuses on **transfer learning** with pretrained CNN backbones.
- You will train **three frozen-backbone models** for a fair comparison:
  - **VGG16** (frozen base)
  - **ResNet50** (frozen base)
  - **DenseNet121** (frozen base)
- Then run **one controlled fine-tuning experiment** (unfreeze a small portion of one backbone with a smaller learning rate).
- Keep your training CPU-feasible (use the recommended settings unless you have a GPU).

## Q1 — Load Dataset & Inspect
Use TFDS: `oxford_iiit_pet` and inspect shapes/classes.
### Student Tasks

- Load the Oxford-IIIT Pet dataset and split into ds_train (80%), ds_val (20%), and ds_test.

- Extract number of classes and class names from ds_info.

- Display one sample: image shape, label index, and class name.

In [2]:
#============================================================
#Question Q1 — Load Oxford-IIIT Pet Dataset (Fill in the Blanks)
#============================================================
#Complete the TODO sections to:
#1) Load the Oxford-IIIT Pet dataset using TensorFlow Datasets
#2) Create train/validation/test splits
#3) Extract class names and number of classes
#4) Print dataset information
#5) Inspect one example image and label
#============================================================

# TODO 1: Load dataset with train/validation/test splits
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train[:80%]", "train[80%:]", "test"],
    as_supervised=True,
    with_info=True,
)

# TODO 2: Get number of classes
num_classes = ds_info.features["label"].num_classes

# TODO 3: Get class names
class_names = ds_info.features["label"].names

# TODO 4: Print dataset information
print("Num classes:", num_classes)
print("Example classes:", class_names[:5])

# TODO 5: Inspect one example from the training set
for x, y in ds_train.take(1):
    print("Raw image shape:", x.shape,
          "| label:", int(y),
          "| class:", class_names[int(y)])

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.HBRPOI_4.0.0/oxford_iiit_pet-train.tfrecord-[0-…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.HBRPOI_4.0.0/oxford_iiit_pet-test.tfrecord-[0-9…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.
Num classes: 37
Example classes: ['Abyssinian', 'american_bulldog', 'american_pit_bull_terrier', 'basset_hound', 'beagle']
Raw image shape: (500, 500, 3) | label: 33 | class: Sphynx


## Q2 — Preprocessing & `tf.data` Pipeline
Resize to **128×128**, normalize, add light augmentation for training.

### Student Tasks

- Set image size and batch size.

- Preprocess images by resizing and normalizing.

- Apply data augmentation to the training dataset.

- Create batched and prefetched train, validation, and test pipelines.

In [3]:
#============================================================
#Question Q2 — Image Preprocessing & Data Pipeline (Fill in the Blanks)
#============================================================
#Complete the TODO sections to:
#1) Define image size (128×128) and batch size (32 to 64)
#2) Implement preprocessing (resize + normalization)
#3) Implement simple data augmentation
#4) Build TensorFlow data pipelines
#5) Shuffle, batch, and prefetch the datasets
#============================================================

# TODO 1: Define image size and batch size
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# ------------------------------------------------------------
# TODO 2: Preprocessing function
# Resize images and normalize pixel values to [0,1]
# ------------------------------------------------------------
@tf.function
def preprocess(image, label):

    # Resize image
    image = tf.image.resize(image, IMG_SIZE)

    # Convert to float32
    image = tf.cast(image, tf.float32)

    # Normalize pixels
    image = image / 255.0

    return image, label

# ------------------------------------------------------------
# TODO 3: Data augmentation function
# ------------------------------------------------------------
@tf.function
def augment(image, label):

    # Random horizontal flip
    image = tf.image.random_flip_left_right(image)

    # Random brightness
    image = tf.image.random_brightness(image, max_delta=0.05)

    return image, label

# ------------------------------------------------------------
# TODO 4: Apply preprocessing and augmentation
# ------------------------------------------------------------
train_ds = ds_train.map(preprocess, num_parallel_calls=AUTOTUNE)\
                   .map(augment, num_parallel_calls=AUTOTUNE)

val_ds   = ds_val.map(preprocess, num_parallel_calls=AUTOTUNE)

test_ds  = ds_test.map(preprocess, num_parallel_calls=AUTOTUNE)

# ------------------------------------------------------------
# TODO 5: Shuffle, batch, and prefetch
# ------------------------------------------------------------
train_ds = train_ds.shuffle(1000, seed=SEED)\
                   .batch(BATCH_SIZE)\
                   .prefetch(AUTOTUNE)

val_ds   = val_ds.batch(BATCH_SIZE)\
                 .prefetch(AUTOTUNE)

test_ds  = test_ds.batch(BATCH_SIZE)\
                  .prefetch(AUTOTUNE)

print("Ready:", train_ds, val_ds, test_ds)

Ready: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))> <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))> <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


## Q3 — Utilities: Compile / Train / Evaluate / Count Params

### Student Tasks

- Compile the model using the Adam optimizer, sparse categorical cross-entropy loss, and accuracy metric.

- Train the model for several epochs and measure the training time.

- Evaluate the trained model on the validation and test datasets and report accuracy.

In [4]:
# ============================================================
# Question Q3 — Model Training Utilities (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Compile a model using the Adam optimizer
# 2) Define the loss function for multi-class classification
# 3) Track accuracy during training
# 4) Compute total model parameters
# 5) Train the model and measure training time
# 6) Evaluate the model on validation and test datasets
# ============================================================

import numpy as np
import time
import tensorflow as tf

# ------------------------------------------------------------
# TODO 1: Compile the model
# ------------------------------------------------------------
def compile_model(model, lr=1e-3):

    model.compile(

        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),

        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),

        metrics=["accuracy"],
    )

    return model


# ------------------------------------------------------------
# TODO 2: Compute total parameters of the model
# ------------------------------------------------------------
def total_params(model):

    return int(np.sum([np.prod(v.shape) for v in model.trainable_variables])) + \
           int(np.sum([np.prod(v.shape) for v in model.non_trainable_variables]))


# ------------------------------------------------------------
# TODO 3: Train the model and measure training time
# ------------------------------------------------------------
def train_and_time(model, run_name, epochs=8):

    t0 = time.time()

    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        verbose=1
    )

    dt = time.time() - t0

    return hist, dt


# ------------------------------------------------------------
# TODO 4: Evaluate the model
# ------------------------------------------------------------
def evaluate(model, name):

    v = model.evaluate(val_ds, verbose=0)

    t = model.evaluate(test_ds, verbose=0)

    print(f"{name} | Val acc: {v[1]:.4f} | Test acc: {t[1]:.4f}")

    return {
        "val_loss": v[0],
        "val_acc": v[1],
        "test_loss": t[0],
        "test_acc": t[1],
    }

## Q4 — Backbones & Transfer Model Builder

We will compare three pretrained ImageNet backbones using the **same head design** (GAP + Dropout + Dense):
- **VGG16**
- **ResNet50**
- **DenseNet121**

For feature extraction, keep the backbone **frozen** (`train_base=False`). For fine-tuning, unfreeze a small number of top layers with a smaller learning rate.

### Student Tasks

- Implement a function to load a pretrained backbone model (VGG16, ResNet50, or DenseNet121).

- Build a transfer learning model by adding a Global Average Pooling and classification layer.

- Optionally unfreeze the last layers of the backbone to perform fine-tuning with a smaller learning rate.


In [5]:
# ============================================================
# Question Q4 — Transfer Learning Model Construction (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a backbone model using pretrained ImageNet weights
# 2) Support VGG16, ResNet50, and DenseNet121 architectures
# 3) Attach a classifier head using Global Average Pooling
# 4) Create a full transfer learning model
# 5) Implement fine-tuning by unfreezing part of the backbone
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build backbone model
# ------------------------------------------------------------
def build_backbone(backbone_name: str):
    """Return (base_model, preprocess_fn) for a supported backbone."""

    name = backbone_name.lower()

    if name == "vgg16":
        base = tf.keras.applications.VGG16(
            weights="imagenet",
            include_top=False,
            input_shape=IMG_SIZE + (3,)
        )

        preprocess_fn = vgg16.preprocess_input

    elif name == "resnet50":
        base = tf.keras.applications.ResNet50(
            weights="imagenet",
            include_top=False,
            input_shape=IMG_SIZE + (3,)
        )

        preprocess_fn = resnet.preprocess_input

    elif name in ["densenet121", "dense121"]:
        base = tf.keras.applications.DenseNet121(
            weights="imagenet",
            include_top=False,
            input_shape=IMG_SIZE + (3,)
        )

        preprocess_fn = densenet.preprocess_input

    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    return base, preprocess_fn


# ------------------------------------------------------------
# TODO 2: Build transfer learning model
# ------------------------------------------------------------
def build_transfer_model(backbone_name="resnet50", train_base=False, dropout=0.2):
    """Backbone + GAP head. Returns (model, base)."""

    base, preprocess_fn = build_backbone(backbone_name)

    # Freeze or unfreeze backbone
    base.trainable = bool(train_base)

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))

    # Apply preprocessing function
    x = preprocess_fn(inputs * 255.0)

    # Forward pass through backbone
    x = base(x, training=False)

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D()(x)

    # Dropout regularization
    x = layers.Dropout(dropout)(x)

    # Output classification layer
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    # Build final model
    model = tf.keras.Model(
        inputs,
        outputs,
        name=f"{backbone_name}_gap_{'ft' if train_base else 'fz'}"
    )

    return model, base


# ------------------------------------------------------------
# TODO 3: Fine-tune the backbone
# ------------------------------------------------------------
def fine_tune(model, base, n_unfreeze=30, lr=1e-5):
    """Unfreeze last layers of backbone and recompile model."""

    # Enable backbone training
    base.trainable = True

    if n_unfreeze is not None:
        for layer in base.layers[:-int(n_unfreeze)]:
            layer.trainable = False

    # Recompile model with smaller learning rate
    model = compile_model(model, lr=lr)

    return model

## Q5 — Model A: **VGG16** Transfer Learning (Frozen Base)
Train only a small classification head first (feature extraction).

### Student Tasks

- Build a transfer learning model using a frozen VGG16 backbone with a Global Average Pooling head.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.

In [6]:
# ============================================================
# Question Q5 — Train VGG16 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a transfer learning model using the VGG16 backbone
# 2) Freeze the backbone during initial training
# 3) Compile the model with the provided compile_model() function
# 4) Train the model and record training time
# 5) Evaluate the trained model on validation and test datasets
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build the VGG16 transfer learning model
# ------------------------------------------------------------
vgg_model, vgg_base = build_transfer_model(
    backbone_name="vgg16",
    train_base=False,
    dropout=0.2
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
vgg_model = compile_model(
    vgg_model,
    lr=1e-3
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
vgg_model.summary()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_vgg_fz, time_vgg_fz = train_and_time(
    vgg_model,
    "VGG16 Frozen",
    epochs=5
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_vgg_fz = evaluate(
    vgg_model,
    "VGG16 Frozen"
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "vgg16_gap_fz"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 128, 128,  │          0 │ input_layer_1[0]… │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 128, 128)  │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 128, 128)  │          0 │ multiply[0][0]    │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 128, 128)  │          0 │ multiply[0][0]    │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 128, 128,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 128,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 4, 4, 512) │ 14,714,688 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 37)        │     18,981 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,733,669 (56.20 MB)

 Trainable params: 18,981 (74.14 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 107ms/step - accuracy: 0.1223 - loss: 19.0816 - val_accuracy: 0.3682 - val_loss: 7.1099
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 98ms/step - accuracy: 0.3791 - loss: 7.9161 - val_accuracy: 0.5571 - val_loss: 4.6598
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 94ms/step - accuracy: 0.5292 - loss: 4.8468 - val_accuracy: 0.6168 - val_loss: 3.8339
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 108ms/step - accuracy: 0.6046 - loss: 3.6224 - val_accuracy: 0.6386 - val_loss: 3.3859
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 88ms/step - accuracy: 0.6678 - loss: 2.9031 - val_accuracy: 0.6671 - val_loss: 3.0995
VGG16 Frozen | Val acc: 0.6671 | Test acc: 0.6470


## Q6 — Model B: **ResNet50** Transfer Learning (Frozen Base)
Same head design as VGG16 model for a fair backbone comparison.

### Student Tasks

- Build a transfer learning model using a frozen ResNet50 backbone with a Global Average Pooling head.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.

In [7]:
# ============================================================
# Question Q6 — Train ResNet50 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a transfer learning model using the ResNet50 backbone
# 2) Freeze the backbone during initial training
# 3) Compile the model using compile_model()
# 4) Train the model and record training time
# 5) Evaluate the trained model on validation and test datasets
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build the ResNet50 transfer learning model
# ------------------------------------------------------------
resnet_model, resnet_base = build_transfer_model(
    backbone_name="resnet50",
    train_base=False,
    dropout=0.2
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
resnet_model = compile_model(
    resnet_model,
    lr=1e-3
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
resnet_model.summary()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_resnet_fz, time_resnet_fz = train_and_time(
    resnet_model,
    "ResNet50 Frozen",
    epochs=5
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_resnet_fz = evaluate(
    resnet_model,
    "ResNet50 Frozen"
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "resnet50_gap_fz"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 128, 128,  │          0 │ input_layer_3[0]… │
│ (Multiply)          │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 128, 128)  │          0 │ multiply_1[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 128, 128)  │          0 │ multiply_1[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 128, 128)  │          0 │ multiply_1[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 128, 128,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 128, 128,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 4, 4,      │ 23,587,712 │ add_1[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 37)        │     75,813 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,663,525 (90.27 MB)

 Trainable params: 75,813 (296.14 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 26s 123ms/step - accuracy: 0.4766 - loss: 2.2627 - val_accuracy: 0.7323 - val_loss: 0.9146
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.7639 - loss: 0.7522 - val_accuracy: 0.7527 - val_loss: 0.8235
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - accuracy: 0.8550 - loss: 0.4228 - val_accuracy: 0.7717 - val_loss: 0.8263
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 75ms/step - accuracy: 0.8923 - loss: 0.3206 - val_accuracy: 0.7908 - val_loss: 0.8126
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 88ms/step - accuracy: 0.9253 - loss: 0.2355 - val_accuracy: 0.7812 - val_loss: 0.8019
ResNet50 Frozen | Val acc: 0.7812 | Test acc: 0.7585


## Q7 — Model C: **DenseNet121** Transfer Learning (Frozen Base)
Train a DenseNet121 backbone with the same GAP head design for a fair comparison against VGG16 and ResNet50.

### Student Tasks

- Build a transfer learning model using a frozen DenseNet121 backbone.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.


In [8]:
# ============================================================
# Question Q7 — Train DenseNet121 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a transfer learning model using the DenseNet121 backbone
# 2) Freeze the backbone during initial training
# 3) Compile the model using compile_model()
# 4) Train the model and measure training time
# 5) Evaluate the trained model on validation and test datasets
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build the DenseNet121 transfer learning model
# ------------------------------------------------------------
densenet_model, densenet_base = build_transfer_model(
    backbone_name="densenet121",    # e.g., densenet121
    train_base=False,
    dropout=0.2
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
densenet_model = compile_model(
    densenet_model,
    lr=1e-3
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
densenet_model.summary()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_densenet_fz, time_densenet_fz = train_and_time(
    densenet_model,
    "DenseNet121 Frozen",
    epochs=5
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_densenet_fz = evaluate(
    densenet_model,
    "DenseNet121 Frozen"
)

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "densenet121_gap_fz"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply_2 (Multiply)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_2 (Add)                     │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 37)             │        37,925 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,075,429 (26.99 MB)

 Trainable params: 37,925 (148.14 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 48s 206ms/step - accuracy: 0.3689 - loss: 2.7338 - val_accuracy: 0.7310 - val_loss: 0.9274
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 89ms/step - accuracy: 0.7031 - loss: 0.9288 - val_accuracy: 0.7894 - val_loss: 0.7045
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 78ms/step - accuracy: 0.7853 - loss: 0.6820 - val_accuracy: 0.8003 - val_loss: 0.6378
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 96ms/step - accuracy: 0.8319 - loss: 0.5293 - val_accuracy: 0.8071 - val_loss: 0.6228
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 103ms/step - accuracy: 0.8580 - loss: 0.4072 - val_accuracy: 0.8315 - val_loss: 0.5620
DenseNet121 Frozen | Val acc: 0.8315 | Test acc: 0.8013


## Q8 — Fine-Tuning Experiment (One Controlled Change)

Fine-tune **all three backbone models (VGG16, ResNet50, and DenseNet121)** by unfreezing only the **top N layers**.

Report whether performance **improves, stays similar, or degrades**, and briefly explain why.

**Recommended**
- Start by unfreezing the **last 10–30 layers**
- Use a **smaller learning rate** (e.g., `1e-5`)
- Use **2-3 epochs**

### Student Tasks

- Fine-tune the **VGG16, ResNet50, and DenseNet121** models by unfreezing the top layers of each backbone.

- Train each fine-tuned model for several epochs.

- Evaluate the validation and test accuracy of each fine-tuned model.

## Q8(a) — Fine-tune VGG16

In [9]:
# ============================================================
# Question Q8(a) — Fine-Tune VGG16 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Fine-tune the VGG16 backbone by unfreezing the last layers
# 2) Re-compile the model with a smaller learning rate
# 3) Train the fine-tuned model
# 4) Evaluate the model after fine-tuning
# ============================================================

# ------------------------------------------------------------
# TODO 1: Fine-tune the VGG16 backbone
# ------------------------------------------------------------
vgg_model = fine_tune(
    vgg_model,
    vgg_base,
    n_unfreeze=20,
    lr=1e-5
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_vgg_ft, time_vgg_ft = train_and_time(
    vgg_model,
    "VGG16 Fine-Tuned",
    epochs=3
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_vgg_ft = evaluate(
    vgg_model,
    "VGG16 Fine-Tuned"
)

Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 56s 235ms/step - accuracy: 0.6732 - loss: 1.4495 - val_accuracy: 0.6739 - val_loss: 1.2818
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 230ms/step - accuracy: 0.7680 - loss: 0.7596 - val_accuracy: 0.6793 - val_loss: 1.2044
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 227ms/step - accuracy: 0.8156 - loss: 0.5779 - val_accuracy: 0.6916 - val_loss: 1.1307
VGG16 Fine-Tuned | Val acc: 0.6916 | Test acc: 0.6765


## Q8(b) — Fine-tune ResNet50

In [10]:
# ============================================================
# Question Q8(b) — Fine-Tune ResNet50 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Fine-tune the ResNet50 backbone by unfreezing the last layers
# 2) Re-compile the model with a smaller learning rate
# 3) Train the fine-tuned model
# 4) Evaluate the model after fine-tuning
# ============================================================

# ------------------------------------------------------------
# TODO 1: Fine-tune the ResNet50 backbone
# ------------------------------------------------------------
resnet_model = fine_tune(
    resnet_model,
    resnet_base,
    n_unfreeze=20,
    lr=1e-5
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_resnet_ft, time_resnet_ft = train_and_time(
    resnet_model,
    "ResNet50 Fine-Tuned",
    epochs=3
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_resnet_ft = evaluate(
    resnet_model,
    "ResNet50 Fine-Tuned"
)

Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 32s 132ms/step - accuracy: 0.9226 - loss: 0.3193 - val_accuracy: 0.7935 - val_loss: 0.7109
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 92ms/step - accuracy: 0.9623 - loss: 0.2221 - val_accuracy: 0.7894 - val_loss: 0.7112
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 80ms/step - accuracy: 0.9752 - loss: 0.1794 - val_accuracy: 0.7880 - val_loss: 0.7031
ResNet50 Fine-Tuned | Val acc: 0.7880 | Test acc: 0.7424


## Q8(c) — Fine-tune DenseNet121

In [11]:
# ============================================================
# Question Q8(c) — Fine-Tune DenseNet121 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Fine-tune the DenseNet121 backbone by unfreezing the last layers
# 2) Re-compile the model with a smaller learning rate
# 3) Train the fine-tuned model
# 4) Evaluate the model after fine-tuning
# ============================================================

# ------------------------------------------------------------
# TODO 1: Fine-tune the DenseNet121 backbone
# ------------------------------------------------------------
densenet_model = fine_tune(
    densenet_model,
    densenet_base,
    n_unfreeze=5,
    lr=1e-5
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_densenet_ft, time_densenet_ft = train_and_time(
    densenet_model,
    "DenseNet121 Fine-Tuned",
    epochs=3
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_densenet_ft = evaluate(
    densenet_model,
    "DenseNet121 Fine-Tuned"
)

Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 46s 209ms/step - accuracy: 0.8505 - loss: 0.5705 - val_accuracy: 0.8139 - val_loss: 0.6049
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 91ms/step - accuracy: 0.8709 - loss: 0.5427 - val_accuracy: 0.8166 - val_loss: 0.6338
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 87ms/step - accuracy: 0.8747 - loss: 0.5285 - val_accuracy: 0.8139 - val_loss: 0.6411
DenseNet121 Fine-Tuned | Val acc: 0.8139 | Test acc: 0.7975


## Q9 — Performance Comparison Table

#### Create a compact table comparing **frozen-backbone** models:

- VGG16 (frozen)
- ResNet50 (frozen)
- DenseNet121 (frozen)

#### Also include **fine-tuned** results for all three backbone.

### Student Tasks

- Create a comparison table summarizing the results of all models.

- Include validation accuracy, test accuracy, and training time.

- Compare frozen and fine-tuned models and identify the best-performing model.


In [12]:
# ============================================================
# Question Q9 — Compare Model Results (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Store evaluation results for frozen models
# 2) Append results for fine-tuned models if they exist
# 3) Create a comparison table using pandas
# 4) Sort models by test accuracy
# ============================================================

# ------------------------------------------------------------
# TODO 1: Create result rows for frozen models
# ------------------------------------------------------------
rows = [

    {"Model": "VGG16 Frozen",
     "Val acc": res_vgg_fz["val_acc"],
     "Test acc": res_vgg_fz["test_acc"],
     "Train time (s)": time_vgg_fz},

    {"Model": "ResNet50 Frozen",
     "Val acc": res_resnet_fz["val_acc"],
     "Test acc": res_resnet_fz["test_acc"],
     "Train time (s)": time_resnet_fz},

    {"Model": "DenseNet121 Frozen",
     "Val acc": res_densenet_fz["val_acc"],
     "Test acc": res_densenet_fz["test_acc"],
     "Train time (s)": time_densenet_fz},
]

# ------------------------------------------------------------
# TODO 2: Append results for fine-tuned models if available
# ------------------------------------------------------------

if "res_vgg_ft" in globals():
    rows.append({
        "Model": "VGG16 Fine-Tuned",
        "Val acc": res_vgg_ft["val_acc"],
        "Test acc": res_vgg_ft["test_acc"],
        "Train time (s)": time_vgg_ft
    })

if "res_resnet_ft" in globals():
    rows.append({
        "Model": "ResNet50 Fine-Tuned",
        "Val acc": res_resnet_ft["val_acc"],
        "Test acc": res_resnet_ft["test_acc"],
        "Train time (s)": time_resnet_ft
    })

if "res_densenet_ft" in globals():
    rows.append({
        "Model": "DenseNet121 Fine-Tuned",
        "Val acc": res_densenet_ft["val_acc"],
        "Test acc": res_densenet_ft["test_acc"],
        "Train time (s)": time_densenet_ft
    })


# ------------------------------------------------------------
# TODO 3: Create comparison dataframe
# ------------------------------------------------------------
df = pd.DataFrame(rows)


# ------------------------------------------------------------
# TODO 4: Sort models by test accuracy
# ------------------------------------------------------------
df = df.sort_values("Test acc", ascending=False)

df

,Model,Val acc,Test acc,Train time (s)
2,DenseNet121 Frozen,0.831522,0.801308,96.630948
5,DenseNet121 Fine-Tuned,0.813859,0.797493,67.015201
1,ResNet50 Frozen,0.781250,0.758517,65.310719
4,ResNet50 Fine-Tuned,0.788043,0.742437,51.816648
3,VGG16 Fine-Tuned,0.691576,0.676479,104.834859
0,VGG16 Frozen,0.667120,0.647043,74.595108


## Short Written Questions

## **Q10** — What is the difference between *feature extraction* and *fine-tuning* in transfer learning?  
Feature extraction means we keep the pretrained backbone frozen and only train the new classifier head for the pet dataset. Fine-tuning goes a step further by unfreezing some of the later backbone layers so the model can adjust more to this specific dataset. Based on my results, feature extraction worked really well for DenseNet121 because the frozen model had the best test accuracy overall. Fine-tuning helped VGG16 a little, but it did not improve DenseNet121 or ResNet50 on the test set.

## **Q11** — Why should the learning rate typically be lower during fine-tuning than during feature extraction?  

The learning rate should be smaller during fine-tuning because the backbone already has useful pretrained ImageNet features. If the learning rate is too large, the model can change those weights too aggressively and lose what it already learned. Using a smaller learning rate lets the model make more careful updates. In my results, fine-tuning did not always improve performance, which shows why it has to be done carefully.

## **Q12** — Give one reason why these backbones may behave differently on Oxford-IIIT Pet.  

The backbones can behave differently because they are designed differently. VGG16 is simpler and uses stacked convolution layers, ResNet50 uses skip connections, and DenseNet121 reuses features through dense connections. In my results, DenseNet121 performed the best, which may be because its feature reuse helped it learn useful pet image features more effectively. VGG16 was the weakest overall, but fine-tuning did improve it slightly.

## **Q13** — Under what conditions can fine-tuning reduce performance compared to a frozen backbone?  

Fine-tuning can reduce performance when the model changes the pretrained weights too much, especially if the dataset is limited or the learning rate is not low enough. It can also overfit because more layers become trainable. In my results, DenseNet121 Frozen slightly outperformed DenseNet121 Fine-Tuned, and ResNet50 Frozen also did better than ResNet50 Fine-Tuned on the test set. That suggests fine-tuning was not always helpful here, and the frozen pretrained features were already strong.

### 🎉 Congratulations!

You have successfully completed **P4-TL**. Excellent work exploring and comparing **RL**, specifically **VGG16**, **VGG16**, and **DenseNet121**. Analyzing how **TL, and fine-tuning** affect performance on the **Oxford-IIIT Pet** under **CPU-only training constraints**.

### **Submission Instructions**

Please submit a **GitHub repository link** on Canvas that contains:
- The **completed Jupyter notebook**
- Notebook runs **top-to-bottom** without errors

Before submitting, ensure that:
- All **code cells (Q1–Q9)** have been executed successfully
- All **Markdown responses (Q10–Q13)** have been completed
- The notebook is **saved after execution** so that outputs are visible

Once verified, **push the final version to GitHub** and submit the repository link on Canvas.